In [0]:
# Padronização de nomes: <catalog>.<camada>.<tabela>, tudo em snake_case
catalog = "cinedata"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

print(f"Catalog: {catalog}")
print(f"Bronze Schema: {bronze_schema}")
print(f"Silver Schema: {silver_schema}")
print(f"Gold Schema: {gold_schema}")
print(f"Landing Path: {landing_path}")


Catalog: cinedata
Bronze Schema: cinedata.bronze
Silver Schema: cinedata.silver
Gold Schema: cinedata.gold
Landing Path: /Volumes/cinedata/bronze/landing


In [0]:
# Criar Catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")

# Criar Schemas (bronze, silver, gold)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

# Criar Volume na zona bronze (aqui vão os CSVs brutos)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.landing")

print("Catalog, schemas e volume criados com sucesso!")


Catalog, schemas e volume criados com sucesso!


In [0]:
expected_files = [
    "movies_info_TMDB_IMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_reviews.csv",
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Verifique se o volume foi criado.")
    print(f"Erro: {e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
    print("\nFaça upload dos CSVs antes de continuar (Catalog → cinedata → bronze → landing → Upload to this volume)")
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")


[OK] Todos os arquivos esperados estão na landing zone:
  - credits_and_tags_IMDB_TMDB.csv (21797.5 KB)
  - movies_financials_IMDB_TMDB.csv (1433.3 KB)
  - movies_info_TMDB_IMDB.csv (33129.0 KB)
  - movies_metrics_IMDB_TMDB.csv (3246.0 KB)
  - movies_reviews.csv (1878.4 KB)


In [0]:
from pyspark.sql.functions import current_timestamp

# Mapeamento dos caminhos no Volume
path_movies_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_movies_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_movies_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_movies_reviews = f"{landing_path}/movies_reviews.csv"

# 1. Leitura PURA (sem inferSchema, tudo como STRING)
print("Lendo CSVs como STRING puro (sem transformações)...")

df_movies_info_raw = spark.read.csv(path_movies_info, header=True, inferSchema=False)
df_movies_financials_raw = spark.read.csv(path_movies_financials, header=True, inferSchema=False)
df_movies_metrics_raw = spark.read.csv(path_movies_metrics, header=True, inferSchema=False)
df_credits_and_tags_raw = spark.read.csv(path_credits_and_tags, header=True, inferSchema=False)
df_movies_reviews_raw = spark.read.csv(path_movies_reviews, header=True, inferSchema=False)

print("Todos os CSVs foram lidos como STRING")


Lendo CSVs como STRING puro (sem transformações)...
Todos os CSVs foram lidos como STRING


In [0]:
# 2. Gravação com adição do timestamp NO MOMENTO DA ESCRITA
# (Por causa da Lazy Evaluation do Spark, o timestamp só é calculado quando a ação .write() dispara)

print("Gravando 5 tabelas Bronze em Delta (modo APPEND)...\n")

# Tabela 1: tb_movies_info
df_movies_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{bronze_schema}.tb_movies_info")
print("tb_movies_info gravada")

# Tabela 2: tb_movies_financials
df_movies_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{bronze_schema}.tb_movies_financials")
print("tb_movies_financials gravada")

# Tabela 3: tb_movies_metrics
df_movies_metrics_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{bronze_schema}.tb_movies_metrics")
print("tb_movies_metrics gravada")

# Tabela 4: tb_credits_and_tags
df_credits_and_tags_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{bronze_schema}.tb_credits_and_tags")
print("tb_credits_and_tags gravada")

# Tabela 5: tb_movies_reviews
df_movies_reviews_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{bronze_schema}.tb_movies_reviews")
print("tb_movies_reviews gravada")

print("\nTodas as 5 tabelas Bronze foram gravadas com sucesso!")


Gravando 5 tabelas Bronze em Delta (modo APPEND)...

tb_movies_info gravada
tb_movies_financials gravada
tb_movies_metrics gravada
tb_credits_and_tags gravada
tb_movies_reviews gravada

Todas as 5 tabelas Bronze foram gravadas com sucesso!


In [0]:
import requests
from datetime import datetime, timedelta
import json

print("Buscando cotação USD/BRL do Banco Central (últimos 7 dias)...\n")

# Data final: hoje
data_fim = datetime.now().strftime("%m-%d-%Y")  # Formato: MM-DD-YYYY

# Data inicial: 7 dias atrás (garante pelo menos 1 dia útil, já que BCB não retorna fins de semana)
data_inicio = (datetime.now() - timedelta(days=7)).strftime("%m-%d-%Y")

print(f"Período: {data_inicio} até {data_fim}\n")

# URL e parâmetros da API PTAX
url = "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo"

params = {
    f"dataInicial='{data_inicio}'": None,
    f"dataFinalCotacao='{data_fim}'": None,
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json"
}

# Construir URL correta (PTAX é um pouco diferente)
url_completo = f"{url}(dataInicial='{data_inicio}',dataFinalCotacao='{data_fim}')?$select=dataHoraCotacao,cotacaoCompra&$format=json"

print(f"URL: {url_completo}\n")

try:
    response = requests.get(url_completo, timeout=10)
    response.raise_for_status()
    
    dados_api = response.json()
    
    # Verificar estrutura da resposta
    if "value" in dados_api:
        registros_api = dados_api["value"]
    else:
        registros_api = dados_api
    
    print(f"API retornou {len(registros_api)} registros")
    
    # Converter para lista de dicts (format esperado pelo Spark)
    records = []
    for item in registros_api:
        records.append({
            "data_cotacao": item.get("dataHoraCotacao", ""),
            "cotacao_usd_brl": float(item.get("cotacaoCompra", 0))
        })
    
    # Criar DataFrame do Spark
    df_cotacao = spark.createDataFrame(records)
    
    print(f"DataFrame criado com {df_cotacao.count()} registros\n")
    
except Exception as e:
    print(f"[ERRO] Falha ao buscar API: {e}")
    print("Criando DataFrame vazio para não interromper o fluxo...")
    df_cotacao = spark.createDataFrame([], "data_cotacao STRING, cotacao_usd_brl FLOAT")


Buscando cotação USD/BRL do Banco Central (últimos 7 dias)...

Período: 09-14-2026 até 09-21-2026

URL: https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial='09-14-2026',dataFinalCotacao='09-21-2026')?$select=dataHoraCotacao,cotacaoCompra&$format=json

API retornou 6 registros
DataFrame criado com 6 registros



In [0]:
# Adicionar ingestion_datetime e gravar
print("Gravando tb_cotacao_dolar em Delta...\n")

df_cotacao \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")

print("tb_cotacao_dolar gravada com sucesso!")


Gravando tb_cotacao_dolar em Delta...

tb_cotacao_dolar gravada com sucesso!


In [0]:
print("VALIDAÇÃO FINAL — CAMADA BRONZE")

tables = [
    "tb_movies_info",
    "tb_movies_financials",
    "tb_movies_metrics",
    "tb_credits_and_tags",
    "tb_movies_reviews",
    "tb_cotacao_dolar"
]

all_ok = True

print("\nContagem de registros por tabela:\n")
for table_name in tables:
    full_table_name = f"{bronze_schema}.{table_name}"
    try:
        count = spark.table(full_table_name).count()
        print(f" {table_name}: {count:,} registros")
    except Exception as e:
        print(f"{table_name}: ERRO — {e}")
        all_ok = False

print("\n Verificando coluna 'ingestion_datetime' em cada tabela:\n")
for table_name in tables:
    full_table_name = f"{bronze_schema}.{table_name}"
    try:
        df = spark.table(full_table_name)
        cols = df.columns
        if "ingestion_datetime" in cols:
            print(f" {table_name}: ingestion_datetime presente")
        else:
            print(f" {table_name}: ingestion_datetime AUSENTE")
            all_ok = False
    except Exception as e:
        print(f" {table_name}: ERRO — {e}")
        all_ok = False

if all_ok:
    print("BRONZE LAYER PRONTA PARA USO")
else:
    print("Algumas verificações falharam. Revise os blocos anteriores.")

VALIDAÇÃO FINAL — CAMADA BRONZE

Contagem de registros por tabela:

 tb_movies_info: 748,510 registros
 tb_movies_financials: 743,155 registros
 tb_movies_metrics: 751,548 registros
 tb_credits_and_tags: 744,240 registros
 tb_movies_reviews: 226,884 registros
 tb_cotacao_dolar: 32 registros

 Verificando coluna 'ingestion_datetime' em cada tabela:

 tb_movies_info: ingestion_datetime presente
 tb_movies_financials: ingestion_datetime presente
 tb_movies_metrics: ingestion_datetime presente
 tb_credits_and_tags: ingestion_datetime presente
 tb_movies_reviews: ingestion_datetime presente
 tb_cotacao_dolar: ingestion_datetime presente
BRONZE LAYER PRONTA PARA USO
